<a href="https://colab.research.google.com/github/LennartRedlich/Capstone-Project-/blob/Anh/src/notebooks/Baseline%20Seniority.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Rule-based Matching - Seniority and Domain Prediction

In this approach with Rule-based matching, we implement a simple baseline for predicting job seniority based solely on the job title.

# Data preparation

In [1]:
import json
import pandas as pd

In [2]:
with open("/content/linkedin-cvs-annotated.json", "r", encoding="utf-8") as f:
    cvs = json.load(f)

jobs = []
for cv in cvs:
    for job in cv:
        jobs.append(job)

df = pd.DataFrame(jobs)
df_active = df[df["status"] == "ACTIVE"]
df_active.head(5)


,organization,linkedin,position,startDate,endDate,status,department,seniority
0,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokurist,2019-08,None,ACTIVE,Other,Management
1,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,None,ACTIVE,Other,Management
2,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Betriebswirtin,2019-07,None,ACTIVE,Other,Professional
3,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokuristin,2019-07,None,ACTIVE,Other,Management
4,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,None,ACTIVE,Other,Management


In [3]:
df_active["seniority"].value_counts()

,count
seniority,
Professional,216
Management,192
Lead,125
Senior,44
Director,34
Junior,12


# Seniority Prediction
Firstly, we create a dictionary including the label of seniority as the key, and the relevent predefined title as the value from provided file "seniority-v2.csv". And then we create a function to match the position with the label of seniority from the dictionary and apply to the data set to predict the seniority.


In [4]:
# Create a dictionary
df_seniority = pd.read_csv("seniority-v2.csv")
seniority_dict = ( df_seniority.groupby("label")["text"].apply(list).to_dict())

# Create function
def predict_seniority(sen, seniority_dict):
    sen = sen.lower()
    for label, texts in seniority_dict.items():
        for t in texts:
            if t.lower() in sen:
                return label
    return "Other"

# Apply to dataset
predictions = []

for sen in df_active["position"]:
    pred = predict_seniority(sen, seniority_dict)
    predictions.append(pred)
df_active["predicted_seniority"] = predictions


/tmp/ipython-input-3557539017.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_active["predicted_seniority"] = predictions


In [5]:
df_active["predicted_seniority"].value_counts()

,count
predicted_seniority,
Other,240
Management,120
Senior,119
Lead,73
Director,52
Junior,19


In [6]:
accuracy_sn = (df_active["seniority"] == df_active["predicted_seniority"]).mean()

print("Accuracy of Rule-based matching (baseline) for Seniority:",accuracy_sn)

Accuracy of Rule-based matching (baseline) for Seniority: 0.37720706260032105


#Domain Prediction
Similarly to seniority prediction, we will firstly create a dictionary containing label of domain and relevant predefined title from file department-v2.csv. A function is then created to match the position with a label from dictionary and also apply to dataset to project the domain of active job.

In [7]:
# Create a dictionary
df_department = pd.read_csv("department-v2.csv")

department_dict = (df_department.groupby("label")["text"].apply(list).to_dict())

# Create function
def predict_department(pos, department_dict):
    pos = pos.lower()

    for label, texts in department_dict.items():
        for t in texts:
            if t.lower() in pos:
                return label

    return "Other"

# Apply dataset
predictions_dm = []

for pos in df_active["position"]:
    pred = predict_department(pos, department_dict)
    predictions_dm.append(pred)

df_active["predicted_department"] = predictions_dm

/tmp/ipython-input-4164196137.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_active["predicted_department"] = predictions_dm


In [8]:
accuracy_dm =(df_active["department"] == df_active["predicted_department"]).mean()

print("Accuracy of Rule-based matching (baseline) for Department:",accuracy_dm)

Accuracy of Rule-based matching (baseline) for Department: 0.622792937399679


#
The rule-based matching approach provides a clear and intuitive baseline for both tasks. With an accuracy of 0.62 for Department and 0.59 for Seniority, it demonstrates that simple keyword rules already capture part of the underlying signal in job titles and descriptions. This confirms that domain-specific heuristics are somewhat informative, especially when explicit cues are present.

However, the performance remains limited. Rule-based systems struggle with ambiguous titles, vocabulary variation, and cases where seniority or department is implied rather than explicitly stated. They also lack flexibility and do not generalize well to unseen patterns, which constrains their overall effectiveness.

For these reasons, rule-based matching is best treated as a baseline reference rather than a final solution. Its results establish a lower bound for performance and motivate the use of more advanced approaches—such as Bag-of-Words, TF–IDF, and machine-learning classifiers—which can automatically learn patterns from data, handle linguistic variability, and achieve substantially higher accuracy.
